In [1]:
import torch
import torch.nn as nn
import einops
from fancy_einsum import einsum
import tqdm.auto as tqdm
import plotly.express as px

from jaxtyping import Float
from functools import partial

/Users/fletcaw1/Documents/Personal/personal-repos/ARENA_3.0/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Transformer Lens Tutorial

In [2]:
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookPoint,
)  # Hooking utilities
from transformer_lens import HookedTransformer, FactoredMatrix

With focus on inference not training we turn off automatic differentiation.

In [3]:
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

Detect the device.

In [4]:
device = utils.get_device()

Some plotting helper functions.

In [5]:
def imshow(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    px.imshow(utils.to_numpy(tensor), color_continuous_midpoint=0.0, color_continuous_scale="RdBu", labels={"x":xaxis, "y":yaxis}, **kwargs).show(renderer)

def line(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    px.line(utils.to_numpy(tensor), labels={"x":xaxis, "y":yaxis}, **kwargs).show(renderer)

def scatter(x, y, xaxis="", yaxis="", caxis="", renderer=None, **kwargs):
    x = utils.to_numpy(x)
    y = utils.to_numpy(y)
    px.scatter(y=y, x=x, labels={"x":xaxis, "y":yaxis, "color":caxis}, **kwargs).show(renderer)

# Loading and Running Models

In [6]:
model = HookedTransformer.from_pretrained("gpt2-small", device=device)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer


We run a small sample and return the loss.

In [7]:
random_text = "The cat sat on the mat"

loss = model(random_text, return_type="loss")
print(f"Model loss", loss)

Model loss tensor(4.5781, device='mps:0')


We could also return:
- "logits"
- "loss"
- "both"
- None (faster when we only care about internal activations)

We want to see the internal activations of the model using `model.run_with_cache(tokens)`.

In [8]:
tokens = model.to_tokens(random_text)
print(f"Tokens: {tokens}")
logits, cache = model.run_with_cache(tokens, remove_batch_dim=True)

Tokens: tensor([[50256,   464,  3797,  3332,   319,   262,  2603]], device='mps:0')


We plot the attention pattern of all the heads in layer 0.

In [9]:
import circuitsvis as cv
# Testing that the library works
cv.examples.hello("Neel")

In [10]:
print(type(cache))
attention_pattern =cache["pattern", 0, "attn"]
print(attention_pattern.shape)
str_tokens = model.to_str_tokens(tokens)

<class 'transformer_lens.ActivationCache.ActivationCache'>
torch.Size([12, 7, 7])


In [11]:
print("Layer 0 Head Attention Patterns:")
cv.attention.attention_patterns(tokens=str_tokens, attention=attention_pattern)

Layer 0 Head Attention Patterns:


We can also look to extract the 5th layer of the residual stream.

In [12]:
layer_5_activations = cache["blocks.5.hook_resid_post"]
print(layer_5_activations.shape)

torch.Size([7, 768])


We see we have a single batch, with 7 tokens and the model has dimension 768.

# Matrix Multiplication of Probes

Consider how the activations of a single token `[512]` need to be transformed into
the two output classes `[2]`.

In [13]:
activation = torch.arange(512)
output = torch.arange(2)

weight = torch.arange(512*2).reshape(2, 512)

print(f"Weight shape: {weight.shape}")

output = weight @ activation

print(f"Output shape: {output.shape}")

Weight shape: torch.Size([2, 512])
Output shape: torch.Size([2])


Collapsing a dimension.

In [20]:
activations = torch.arange(16*20*128).reshape(16, 20, 128)

interesting_tokens = torch.arange(16)

interesting_activations = activations[torch.arange(16), interesting_tokens]

interesting_activations.shape

torch.Size([16, 128])

Finding a truth vector.

In [22]:
true_activations = torch.arange(32*512).reshape(32, 512).float()
false_activations = torch.arange(32*512).reshape(32, 512).float() + 1000

diff = true_activations - false_activations

diff = torch.mean(diff, dim=0)

diff.shape

torch.Size([512])

Using einsum to fo matrix multiplication.

In [28]:
x = torch.arange(8*1024).reshape(8, 1024)
W = torch.arange(2*1024).reshape(1024, 2)

y = torch.einsum("b d, d o -> b o", x, W)

y.shape

torch.Size([8, 2])

# Training a simple probe

In [ ]:
d_model = layer_5_activations.shape[-1]
n_classes = 2

n_epochs

probe = nn.Linear(d_model, n_classes).to(device)
optimizer = torch.optim.Adam(probe.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(n_epochs):
    logits = probe(activations)
    loss = criterion(logits, labels)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

NameError: name 'n_epochs' is not defined